# GraphCast runs -- spatial generalization experiments

Produces the 2m-temperature forecast-error arrays that `plot_forecast_errors.ipynb`
turns into four of the paper's figures. ERA5 preparation lives in
`prepare_era5_input.ipynb`; the TOA and datetime perturbation experiments are a
separate set of runs and live in `perturbation_experiments.ipynb`.

The experiment compares four test cases. Each one is **365 separate forecasts**
-- one initialised on each day of 2022, each a 40-step six-hourly rollout
covering 10 days -- not a single run spanning the year. The test-case names
below are the canonical ones, used on Zenodo and in the paper; the output
directories keep the original working spellings:

| Test case | Transformation | Output subdirectory |
|---|---|---|
| `baseline` | none | `not_flipped/` |
| `revlon` | longitude reversed | `axis_flip/incl_flipped_wind/` |
| `rotlon` | longitude rotated by 180 deg, with the accompanying -12 h shift of `datetime` | `shift_180/minus_12h/` |
| `revlat` | latitude reversed | `equatorial_flip/incl_flipped_wind/` |

The cells below run one test case at a time; the "Run a variant" note further
down says which values to set for each. All four test cases' output arrays are
available on Zenodo, and `plot_forecast_errors.ipynb` reads those directly, so this
notebook does not need to be run to reproduce any figure.
notebook does not need to be run to reproduce any figure.

Regeneration is gated behind `REGENERATE_FROM_SCRATCH` (default `False`) and
needs a GPU. Note that the results will not match the provided arrays
bit-for-bit, for the reasons set out in the note further down.

Portions of this notebook are adapted from Google DeepMind's GraphCast demo
(<https://github.com/google-deepmind/weathernext>, formerly
`github.com/deepmind/graphcast`) -- the imports, the model construction/wrapping helpers
(`construct_wrapped_graphcast`, `run_forward`, `with_configs`, `with_params`,
`drop_state`) and the `rollout.chunked_prediction` call.
**That code has been modified for this project.** The full Apache License 2.0
text is in `LICENSE-APACHE-2.0` in this folder.

> <p><small><small>Copyright 2023 DeepMind Technologies Limited.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>


## Config

In [ ]:
import sys, os
sys.path.insert(0, "../common")
from zenodo_fetch import ensure_zenodo_files, ensure_forecast_files

DATA_DIR = "../data/GraphCast"
PREDICTIONS_DIR = f"{DATA_DIR}/predictions"

# Canonical test-case name -> (output subdirectory, output filename prefix).
# The directories and filenames keep the original working spellings, so this is
# the one place they appear: set `flip_name` to a canonical name anywhere below
# and the paths follow from here. Note the second path segment is not the same
# for every case -- rotlon's records its -12 h datetime shift, not a wind flip.
CASE_PATHS = {
    "baseline": ("not_flipped",                       "not_flipped"),
    "revlat":   ("equatorial_flip/incl_flipped_wind", "equatorial_flip"),
    "revlon":   ("axis_flip/incl_flipped_wind",       "axis_flip"),
    "rotlon":   ("shift_180/minus_12h",               "shift_180"),
}

# Set True to re-run the revlat forecasts and regenerate its prediction/error
# files instead of using the provided ones. Heavy: 365 ten-day forecasts on a
# GPU, one per day of the year. revlat is the only variant this notebook can
# regenerate -- see the note at the top.
REGENERATE_FROM_SCRATCH = False

# --- TEMPORARY: data is on Zenodo's sandbox while the paper is under review ---
# Sandbox records are periodically wiped and their DOIs (10.5072/...) are not
# real. When the record is published on the real Zenodo, delete these two lines
# and replace the record id(s) below with the real one(s).
import os
os.environ["ZENODO_BASE_URL"] = "https://sandbox.zenodo.org"
# -----------------------------------------------------------------------------

# Zenodo record holding the regridded ERA5 and climatology data.
ZENODO_RECORD_GRAPHCAST_ERA5 = "586387"

# The daily forecasts, one record per test case (~24.9GB each -- a Zenodo record
# is capped at 50GB, so they cannot share one). Only needed if you set
# REGENERATE_FROM_SCRATCH; the error arrays they produce are provided ready-made.
# Within a record the days are bundled into one zip per calendar month, because
# a record also holds at most 100 files and each case has 366 days. The JSON
# beside this notebook maps days to archives and holds every checksum.
FORECAST_CHECKSUMS = "forecast_checksums.json"
# Which test cases to download below. 'baseline' is always fetched -- the
# second error cell differences against it. Each extra case is ~24.9GB.
FETCH_TEST_CASES = ["revlat"]
ZENODO_RECORD_GRAPHCAST_FC_BASELINE = "586998"
ZENODO_RECORD_GRAPHCAST_FC_REVLAT = "587000"
ZENODO_RECORD_GRAPHCAST_FC_REVLON = "587002"
ZENODO_RECORD_GRAPHCAST_FC_ROTLON = "587004"

## Setup (adapted from the GraphCast Demo)

In [ ]:
import dataclasses
import functools
import gc
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from google.cloud import storage
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
import haiku as hk
import jax
import numpy as np
import xarray

from argumentation_functions import flip, flip_back

def parse_file_parts(file_name):
    return dict(part.split("-", 1) for part in file_name.split("_"))

In [ ]:
gcs_client = storage.Client.create_anonymous_client()
gcs_bucket = gcs_client.get_bucket("dm_graphcast")
dir_prefix = "graphcast/"

In [ ]:
with gcs_bucket.blob(dir_prefix + "stats/diffs_stddev_by_level.nc").open("rb") as f:
    diffs_stddev_by_level = xarray.load_dataset(f).compute()
with gcs_bucket.blob(dir_prefix + "stats/mean_by_level.nc").open("rb") as f:
    mean_by_level = xarray.load_dataset(f).compute()
with gcs_bucket.blob(dir_prefix + "stats/stddev_by_level.nc").open("rb") as f:
    stddev_by_level = xarray.load_dataset(f).compute()

### Load the model

Loads the **GraphCast_small** checkpoint (resolution 1.0, mesh size 5, 13 pressure
levels). The checkpoint's own embedded description text says "37 pressure levels",
which is a known inconsistency in the published metadata -- the functional
configuration is 13 levels.

In [ ]:
PARAMS_FILE = "GraphCast_small - ERA5 1979-2015 - resolution 1.0 - pressure levels 13 - mesh 2to5 - precipitation input and output.npz"

with gcs_bucket.blob(f"{dir_prefix}params/{PARAMS_FILE}").open("rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
params = ckpt.params
state = {}
model_config = ckpt.model_config
task_config = ckpt.task_config
print("Model description:\n", ckpt.description)
model_config

## Build jitted functions

In [ ]:
def construct_wrapped_graphcast(
    model_config: graphcast.ModelConfig,
    task_config: graphcast.TaskConfig):
  """Constructs and wraps the GraphCast Predictor."""
  # Deeper one-step predictor.
  predictor = graphcast.GraphCast(model_config, task_config)

  # Modify inputs/outputs to `graphcast.GraphCast` to handle conversion to
  # from/to float32 to/from BFloat16.
  predictor = casting.Bfloat16Cast(predictor)

  # Modify inputs/outputs to `casting.Bfloat16Cast` so the casting to/from
  # BFloat16 happens after applying normalization to the inputs/targets.
  predictor = normalization.InputsAndResiduals(
      predictor,
      diffs_stddev_by_level=diffs_stddev_by_level,
      mean_by_level=mean_by_level,
      stddev_by_level=stddev_by_level)

  # Wraps everything so the one-step model can produce trajectories.
  predictor = autoregressive.Predictor(predictor, gradient_checkpointing=True)
  return predictor


@hk.transform_with_state
def run_forward(model_config, task_config, inputs, targets_template, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  return predictor(inputs, targets_template=targets_template, forcings=forcings)


@hk.transform_with_state
def loss_fn(model_config, task_config, inputs, targets, forcings):
  predictor = construct_wrapped_graphcast(model_config, task_config)
  loss, diagnostics = predictor.loss(inputs, targets, forcings)
  return xarray_tree.map_structure(
      lambda x: xarray_jax.unwrap_data(x.mean(), require_jax=True),
      (loss, diagnostics))

def grads_fn(params, state, model_config, task_config, inputs, targets, forcings):
  def _aux(params, state, i, t, f):
    (loss, diagnostics), next_state = loss_fn.apply(
        params, state, jax.random.PRNGKey(0), model_config, task_config,
        i, t, f)
    return loss, (diagnostics, next_state)
  (loss, (diagnostics, next_state)), grads = jax.value_and_grad(
      _aux, has_aux=True)(params, state, inputs, targets, forcings)
  return loss, diagnostics, next_state, grads

# Jax doesn't seem to like passing configs as args through the jit. Passing it
# in via partial (instead of capture by closure) forces jax to invalidate the
# jit cache if you change configs.
def with_configs(fn):
  return functools.partial(
      fn, model_config=model_config, task_config=task_config)

# Always pass params and state, so the usage below are simpler
def with_params(fn):
  return functools.partial(fn, params=params, state=state)

# Our models aren't stateful, so the state is always empty, so just return the
# predictions. This is requiredy by our rollout code, and generally simpler.
def drop_state(fn):
  return lambda **kw: fn(**kw)[0]

## Load the regridded ERA5 data

Downloaded from Zenodo. See `prepare_era5_input.ipynb` for how this file was produced.

In [ ]:
ensure_zenodo_files(ZENODO_RECORD_GRAPHCAST_ERA5, {
    "regridded_era5_data_for_GraphCast.nc": "a631b97f74850cae3a2855853a548527",
}, DATA_DIR)
reduced_era5_coarsened_test = xarray.open_dataset(f"{DATA_DIR}/regridded_era5_data_for_GraphCast.nc", engine="netcdf4")

## Prep

In [ ]:
days = reduced_era5_coarsened_test.time[:1464].isel(time=slice(0, None, 4))

In [ ]:
all_times = reduced_era5_coarsened_test.time

In [ ]:
reduced_era5_coarsened_test_not_flipped = reduced_era5_coarsened_test.copy()

## Running the model & calculating errors

In [ ]:
def run_model(input_data, path, name):

    train_steps = 1  
    eval_steps = 40  

    train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
        input_data, target_lead_times=slice("6h", f"{train_steps*6}h"),
        **dataclasses.asdict(task_config))

    eval_inputs, eval_targets, eval_forcings = data_utils.extract_inputs_targets_forcings(
        input_data, target_lead_times=slice("6h", f"{eval_steps*6}h"),
        **dataclasses.asdict(task_config))

    
    init_jitted = jax.jit(with_configs(run_forward.init))

    #if params is None:
      #params, state = init_jitted(
          #rng=jax.random.PRNGKey(0),
          #inputs=train_inputs,
          #targets_template=train_targets,
          #forcings=train_forcings)
    
    loss_fn_jitted = drop_state(with_params(jax.jit(with_configs(loss_fn.apply))))
    grads_fn_jitted = with_params(jax.jit(with_configs(grads_fn)))
    run_forward_jitted = drop_state(with_params(jax.jit(with_configs(
        run_forward.apply))))
    
    predictions = rollout.chunked_prediction(
            run_forward_jitted,
            rng=jax.random.PRNGKey(0),
            inputs=eval_inputs,
            targets_template=eval_targets * np.nan,
            forcings=eval_forcings)

    assert model_config.resolution in (0, 360. / eval_inputs.sizes["lon"]), (
          "Model resolution doesn't match the data resolution. You likely want to "
          "re-filter the dataset list, and download the correct data.")

    variables_to_drop = ['temperature', 'geopotential', 'u_component_of_wind', 'v_component_of_wind']
    predictions = predictions.drop_vars(variables_to_drop)
    # Save surface variables of predictions
    surface_vars = predictions.sel(level=1000, method="nearest")
    surface_file = f"{path}{name}_surface_predictions.nc"
    surface_vars.to_netcdf(surface_file)

### Run a variant

The cells below are set up for `revlat`. To produce one of the other three,
change `flip_name` in the first cell and re-run this section -- that is the only
edit needed. The transformations are applied by `flip()` in
`argumentation_functions.py`, and the output directory and filename prefix
follow from `CASE_PATHS` in the Config cell.

| `flip_name` | What `flip()` does |
|---|---|
| *(skip the `flip()` call entirely)* | baseline -- no transformation |
| `revlon` | longitude reversed; sign of the u-wind components flipped |
| `rotlon` | longitude rolled by 180 deg; `datetime` shifted by **-12 h** |
| `revlat` | latitude reversed; sign of the v-wind components flipped; `datetime` shifted by **+183 days** |

Two of the three also move `datetime`, to keep the input physically consistent
with the transformed field. `rotlon` subtracts 12 h because rolling longitude by
180 deg moves every point half a day around the diurnal cycle -- that is what the
`minus_12h/` directory name records. `revlat` adds 183 days because reversing
latitude swaps the hemispheres and the season has to move with them. Half a year
is 182.5 days, but the shift is deliberately a **whole** number of days: half a
day would also move the time of day, reintroducing exactly the diurnal offset
`rotlon` has to correct for. 183 moves the season without touching the clock.
That shift is not recorded in a directory name, as `revlat` and `revlon` share
`incl_flipped_wind/`, which refers to the wind-sign change.

Run the baseline variant before the error cells at the end of this section: the
last cell compares each variant against baseline's per-day predictions.


> **If you set `REGENERATE_FROM_SCRATCH = True`, your output will not be
> bit-identical to the provided `.npy` files -- and two runs of your own will not
> be bit-identical to each other either, unless you force deterministic GPU
> kernels. This is expected, and the cause is known.**
>
> Re-running a single day (`d = 301`) of the two cells below and comparing against
> the provided `error_2mT_era5_corrected.npy`:
>
> | Comparison | RMS difference |
> |---|---|
> | Regenerated vs. provided | 0.126--0.132 K |
> | **Two runs of these cells against each other** | **0.140 K** |
>
> measured against a field whose own RMS is 3.59 K -- agreement to about 3.5%,
> with a correlation of 0.9993.
>
> The run-to-run scatter is *larger* than the regenerated-vs-provided scatter, so a
> fresh rollout is statistically indistinguishable from re-running the original.
>
> A difference of 0.13 K looks big until it is compared against the right
> reference, which is not zero but the model's own reproducibility. It is also far
> too small to be a misalignment, the obvious candidate for a systematic fault: one
> 6 h step of real change in this field is **1.40 K**, and differencing the
> regenerated day against the provided one *shifted by a step* gives 1.41 K, eleven
> times what is actually observed. A misaligned or otherwise wrong input would
> also be at full size in the very first step, whereas the observed difference at
> +6 h is 0.019 K against a 0.98 K field, and only grows thereafter.
>
> ### Where the variation comes from
>
> Not from the model. GraphCast is deterministic and nothing here samples: the
> rollout is driven with a fixed `rng=jax.random.PRNGKey(0)`, and the key is
> passed only because Haiku's `apply` signature requires one. The variation is in
> the GPU kernels underneath.
>
> GraphCast is a graph network, so its central operation is aggregating edge
> messages into nodes -- a scatter-add, implemented on GPU with atomics whose
> accumulation order follows thread scheduling and therefore changes between
> launches. Floating-point addition is not associative, so a different order gives
> different last bits, and `casting.Bfloat16Cast` runs the predictor in bfloat16
> (8 mantissa bits) which makes each rounding coarse. A 40-step autoregressive
> rollout then feeds its own output back in and amplifies the difference.
>
> That was measured rather than assumed (2026-09-09, A100, jax 0.4.29):
>
> | | run A vs run B |
> |---|---|
> | Default kernels | 0.140 K, not identical |
> | `XLA_FLAGS=--xla_gpu_deterministic_ops=true` | **0.000 K, bitwise identical** |
>
> and the same flag takes an isolated `segment_sum` over bfloat16 messages from
> 0/8 to 8/8 bitwise-identical repeats. So the rollout *can* be made exactly
> reproducible, at roughly 10x the runtime -- one day went from 67 s to about
> 11 minutes.
>
> **Determinism does not reproduce the archive, though.** The provided arrays were
> made with default kernels, and deterministic kernels sum in a different order,
> landing elsewhere on the same chaotic trajectory: a deterministic run differs
> from the provided array by 0.185 K, which is *more* than the 0.132 K a default
> run gives. Forcing determinism buys reproducibility from that point on, not
> agreement with what is already archived.
>
> Bit-exact reproduction of the archive is therefore not achievable, and is not the
> right target; agreement to within the run-to-run noise floor is. The SpeedyWeather
> notebooks run into a comparable limit by a different mechanism -- sensitivity to
> CPU architecture, thread count and FFT library *across* machines, rather than GPU
> non-determinism *within* one. **The provided `.npy` arrays are the ones the
> paper's figures were made from; use those to reproduce the published figures.**


In [ ]:
if REGENERATE_FROM_SCRATCH:
    import psutil

    flip_name = "revlat"   # canonical test-case name; paths follow from CASE_PATHS
    case_dir, case_prefix = CASE_PATHS[flip_name]
    data_to_run = reduced_era5_coarsened_test_not_flipped.copy()
    data_to_run = flip(data_to_run, flip_name)

    os.makedirs(f"{PREDICTIONS_DIR}/{case_dir}/", exist_ok=True)

    for d, day in enumerate(days[:365]):
        start = d * 4
        end = start + 42

        current_data = data_to_run.isel(time=slice(start, end)).load()

        elapsed_time = current_data['time'] - current_data['time'].isel(time=0)
        current_data = current_data.assign_coords(time=elapsed_time)

        print(f"{d}, Mem: {psutil.Process().memory_info().rss / 1e9:.2f} GB")
        run_model(
            current_data,
            f'{PREDICTIONS_DIR}/{case_dir}/',
            f'{case_prefix}_{d}'
        )

        del current_data
        gc.collect()

### Computing 2m-temperature errors against ERA5 and against the `baseline` test case


The two cells below need the daily forecast files, not just the rollout this
notebook can regenerate: the first differences a test case against ERA5, the
second against the untransformed **baseline** case, which this notebook does
not produce.

`revlat` is the only case this notebook can also regenerate from scratch — the
code that produced `revlon` and `rotlon` was overwritten and is not recoverable.
Their forecasts are archived all the same, so the differencing below still works
on them if you set `flip_name` accordingly and add them to `FETCH_TEST_CASES`.

Each test case is stored as thirteen monthly zip archives; only the months
holding days you do not already have are downloaded, and each archive is
verified, unpacked and then deleted. So if you ran the rollout above only the
baseline comes down (~24.9GB); if you skipped it, both do (~49.8GB).


In [ ]:
if REGENERATE_FROM_SCRATCH:
    # always needed: the second error cell differences against it
    ensure_forecast_files(ZENODO_RECORD_GRAPHCAST_FC_BASELINE, "baseline",
                          DATA_DIR, FORECAST_CHECKSUMS)

    if "revlat" in FETCH_TEST_CASES:
        ensure_forecast_files(ZENODO_RECORD_GRAPHCAST_FC_REVLAT, "revlat",
                              DATA_DIR, FORECAST_CHECKSUMS)
    if "revlon" in FETCH_TEST_CASES:
        ensure_forecast_files(ZENODO_RECORD_GRAPHCAST_FC_REVLON, "revlon",
                              DATA_DIR, FORECAST_CHECKSUMS)
    if "rotlon" in FETCH_TEST_CASES:
        ensure_forecast_files(ZENODO_RECORD_GRAPHCAST_FC_ROTLON, "rotlon",
                              DATA_DIR, FORECAST_CHECKSUMS)


In [ ]:
if REGENERATE_FROM_SCRATCH:
    variable = '2m_temperature'
    flip_name = "revlat"   # canonical test-case name; paths follow from CASE_PATHS
    case_dir, case_prefix = CASE_PATHS[flip_name]

    errors = np.zeros(shape=(365, 40, 181, 360), dtype=np.float32)
    for d in range(0,365):
        start = d * 4
        end = start + 42

        reference = reduced_era5_coarsened_test_not_flipped.isel(time=slice(start, end)).load()

        _,eval_targets, _ = data_utils.extract_inputs_targets_forcings(
                reference.sel(batch=0), target_lead_times=slice("6h", f"{40 *6}h"),
                **dataclasses.asdict(task_config))

        current_pred = xarray.open_dataset(f'{PREDICTIONS_DIR}/{case_dir}/{case_prefix}_{d}_surface_predictions.nc', engine="netcdf4")

        current_pred = current_pred[variable].sel(batch=0)
        current_pred = flip_back(current_pred, flip_name)

        difference = current_pred.squeeze().values -  eval_targets[variable]

        errors[d, :, :, :] = difference.values
        del reference, eval_targets, current_pred, difference
        gc.collect()

    np.save(f'{PREDICTIONS_DIR}/{case_dir}/error_2mT_era5_corrected.npy', errors)

In [ ]:
if REGENERATE_FROM_SCRATCH:
    baseline_dir, baseline_prefix = CASE_PATHS["baseline"]

    errors_baseline = np.zeros(shape=(365, 40, 181, 360), dtype=np.float32)
    for d in range(0,365):
        start = d * 4
        end = start + 42

        reference = xarray.open_dataset(f'{PREDICTIONS_DIR}/{baseline_dir}/{baseline_prefix}_{d}_surface_predictions.nc', engine="netcdf4")

        reference = reference[variable].sel(batch=0)

        current_pred = xarray.open_dataset(f'{PREDICTIONS_DIR}/{case_dir}/{case_prefix}_{d}_surface_predictions.nc', engine="netcdf4")

        current_pred = current_pred[variable].sel(batch=0)
        current_pred = flip_back(current_pred, flip_name)

        difference = current_pred.squeeze().values -  reference.squeeze().values

        errors_baseline[d, :, :, :] = difference
        del reference, current_pred, difference
        gc.collect()

    np.save(f'{PREDICTIONS_DIR}/{case_dir}/error_2mT_era5_corrected_baseline.npy', errors_baseline)

## The output arrays are provided

All seven prediction-error arrays from this experiment -- each of the three
transforms as a corrected/baseline pair, plus the standalone `baseline` array,
which has no corrected counterpart -- are available on Zenodo and are downloaded
directly by `plot_forecast_errors.ipynb`, so this notebook does not need to be run
to reproduce the figures. File names and checksums are listed in `README.md`.
